<img src="https://raw.githubusercontent.com/telmomm/CERTA/main/docs/assets/logo/certa-logo.png" alt="CERTA" width="220">

# T0.2 · Search and deduplication

*From the database exports of blocks S1 and S2 and the T0.4 tracking to the corpus for screening*

Reads the database exports of the two search blocks of the protocol (S1, test design; S2, safety coverage) and those of the citation and author tracking of T0.4, removes front matter and duplicates, checks the control set and writes the corpus for screening.

- **Input:** one folder per query, as listed in `search_log.csv` (`Scopus/S1`, `WOS/S1`, `WOS/S2_original`, `IEEE/X1`… `IEEE/X4`); the `query_*` folders of T0.4; `../T0.1-protocol/control_set.csv`. A query whose folder is missing or empty is reported as pending.
- **Output:** `corpus.csv` (one row per record, with the databases and blocks it comes from) and `counts.csv` (the numbers of the PRISMA diagram).
- **To re-run:** add or replace the exports, update `search_log.csv` and run all cells. Identifiers `R0001`… already assigned are kept.
- **Environment:** the repository `.venv` (see `CONTRIBUTING.md`).

In [1]:
import warnings
warnings.filterwarnings("ignore")
import re, unicodedata
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 90)
CERTA = {"navy": "#0B2D4A", "teal": "#00BFA5", "blue": "#00AEEF", "sky": "#A7E3F7", "ice": "#E6EEF6"}

class table:
    """Shows a DataFrame with the CERTA colours in the notebook and as a LaTeX table in the PDF export.
    `bar` is a numeric column drawn as bars."""
    def __init__(self, df, bar=None):
        self.df, self.bar = df, bar
    def _repr_html_(self):
        styler = self.df.style.hide(axis="index").set_table_styles([
            {"selector": "th", "props": f"background-color: {CERTA['navy']}; color: white; text-align: left;"},
            {"selector": "tbody tr:nth-child(even)", "props": f"background-color: {CERTA['ice']};"},
        ])
        if self.bar is not None:
            styler = styler.bar(subset=[self.bar], color=CERTA["teal"], vmin=0)
        return styler.to_html()
    def _repr_latex_(self):
        return self.df.style.hide(axis="index").format(precision=3).to_latex(hrules=True)
    def __repr__(self):
        return self.df.to_string(index=False)

CONTROL = Path("../T0.1-protocol/control_set.csv")
CORPUS, COUNTS = Path("corpus.csv"), Path("counts.csv")
T04 = Path("../T0.4-citation-and-author-tracking")
T04_SOURCES = {"backward_citations": "query_backward_citations", "forward_citations": "query_forward_citations", "author": "query_authors"}
DATABASES = ["scopus", "wos", "ieee"]
# (database, query, block, folder): one folder per query, as in search_log.csv
QUERIES = [
    ("scopus", "S1", "S1", "Scopus/S1"), ("wos", "S1", "S1", "WOS/S1"),
    ("ieee", "X1", "S1", "IEEE/X1"), ("ieee", "X2", "S1", "IEEE/X2"),
    ("scopus", "S2", "S2", "Scopus/S2"), ("wos", "S2", "S2", "WOS/S2"), ("wos", "S2_original", "S2", "WOS/S2_original"),
    ("ieee", "X3", "S2", "IEEE/X3"), ("ieee", "X4", "S2", "IEEE/X4"),
]
S2_LIMIT = 1500  # protocol: above this many new records from S2, diagnos* is removed from S2 and the search repeated

## 1. Loading the exports

Every query of the protocol is read from its own folder. Scopus and the T0.4 tracking share the Scopus export format; Web of Science is read from its `.xls` exports and IEEE Xplore from its `.csv` ones.

In [2]:
def read_files(folder, pattern):
    files = sorted(Path(folder).glob(pattern))
    if not files:
        return None
    read = pd.read_excel if pattern.endswith("xls") else lambda f, dtype: pd.read_csv(f, dtype=dtype, encoding="utf-8-sig")
    return pd.concat([read(f, dtype=str) for f in files]).fillna("")

def join(*columns):
    return pd.concat(columns, axis=1).apply(lambda r: "; ".join(v for v in r if v), axis=1)

def from_scopus(df):
    return pd.DataFrame({
        "db_id": df["EID"], "title": df["Title"], "authors": df["Authors"], "year": df["Year"],
        "source": df["Source title"], "type": df["Document Type"],
        "language": df["Language of Original Document"], "doi": df["DOI"], "abstract": df["Abstract"],
        "keywords": join(df["Author Keywords"], df["Index Keywords"]),
    })

def from_wos(df):
    return pd.DataFrame({
        "db_id": df["UT (Unique WOS ID)"], "title": df["Article Title"], "authors": df["Authors"],
        "year": df["Publication Year"].str.split(".").str[0], "source": df["Source Title"],
        "type": df["Document Type"], "language": df["Language"], "doi": df["DOI"], "abstract": df["Abstract"],
        "keywords": join(df["Author Keywords"], df["Keywords Plus"]),
    })

def from_ieee(df):
    return pd.DataFrame({
        "db_id": df["PDF Link"], "title": df["Document Title"], "authors": df["Authors"],
        "year": df["Publication Year"], "source": df["Publication Title"], "type": df["Document Identifier"],
        "language": "", "doi": df["DOI"], "abstract": df["Abstract"],
        "keywords": join(df["Author Keywords"], df["IEEE Terms"]),
    })

READERS = {"scopus": ("*.csv", from_scopus), "wos": ("*.xls", from_wos), "ieee": ("*.csv", from_ieee)}
parts, pending = [], []
for database, query, block, folder in QUERIES:
    pattern, convert = READERS[database]
    df = read_files(folder, pattern)
    if df is None:
        pending.append(f"{database} {query}")
        continue
    parts.append(convert(df).assign(database=database, query=query, block=block))
for database, folder in T04_SOURCES.items():
    df = read_files(T04 / folder, "*.csv")
    if df is not None:
        parts.append(from_scopus(df).assign(database=database, query="tracking", block="tracking"))

raw = pd.concat(parts, ignore_index=True)
raw["doi"] = raw["doi"].str.strip().str.lower().str.replace(r"^https?://(dx\.)?doi\.org/", "", regex=True)
raw["abstract"] = raw["abstract"].replace("[No abstract available]", "")

print("Pending queries:", ", ".join(pending) or "none")
table(raw.groupby(["block", "database", "query"], sort=False).size().rename("records").reset_index())

Pending queries: none


block,database,query,records
S1,scopus,S1,1380
S1,wos,S1,801
S1,ieee,X1,545
S1,ieee,X2,488
S2,scopus,S2,94
S2,wos,S2,52
S2,wos,S2_original,48
S2,ieee,X3,12
S2,ieee,X4,31
tracking,forward_citations,tracking,692


## 2. Front matter and proceedings

IEEE Xplore returns front covers, tables of contents, whole proceedings volumes and standards. Fixed rule: every record without authors is dropped. One more applies to the tracking records: only articles, conference papers, reviews and book chapters enter, as in the Scopus strings.

In [3]:
front_matter = raw["authors"].str.strip().isin(["", "[No author name available]"])
other_type = (raw["query"] == "tracking") & ~raw["type"].isin(["Article", "Conference paper", "Review", "Book chapter"]) & ~front_matter
print("Tracking, dropped by document type:", int(other_type.sum()))
print(front_matter.groupby(raw["database"]).sum().rename("dropped for having no authors").to_string())
rec = raw[~front_matter & ~other_type].reset_index(drop=True)

Tracking, dropped by document type: 7
database
author                0
forward_citations     0
ieee                 71
scopus                3
wos                   0


## 3. Deduplication

1. Same DOI: same record.
2. Same normalised title, same or adjacent year and no conflicting DOIs: same record.
3. Same title with different DOIs: **not** merged. They are flagged with a `possible_version` group (conference and journal version, or a mistyped DOI) and decided at screening, reason E6.

In [4]:
def title_key(t):
    t = re.sub(r"\[[^\]]*\]\s*$", "", t)                      # original title in square brackets (Scopus)
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]", "", t)

rec["title_key"] = rec["title"].map(title_key)
rec["year_n"] = pd.to_numeric(rec["year"], errors="coerce")

parent = list(range(len(rec)))
def root(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def merge(i, j):
    parent[root(i)] = root(j)

# Step 1: by DOI
for _, idx in rec[rec["doi"] != ""].groupby("doi").groups.items():
    for i in idx[1:]:
        merge(i, idx[0])

# Step 2: by title, if the DOIs do not conflict and the year is compatible
def dois(group):
    return {rec.at[i, "doi"] for i in group if rec.at[i, "doi"]}
def years(group):
    return {rec.at[i, "year_n"] for i in group if pd.notna(rec.at[i, "year_n"])}

for key, idx in rec[rec["title_key"].str.len() >= 25].groupby("title_key").groups.items():
    idx = list(idx)
    for a in idx:
        for b in idx:
            if root(a) == root(b):
                continue
            ga = [i for i in idx if root(i) == root(a)]
            gb = [i for i in idx if root(i) == root(b)]
            da, db = dois(ga), dois(gb)
            ya, yb = years(ga), years(gb)
            compatible_doi = not (da and db)
            compatible_year = not (ya and yb) or min(abs(p - q) for p in ya for q in yb) <= 1
            if compatible_doi and compatible_year:
                merge(a, b)

rec["group"] = [root(i) for i in range(len(rec))]
print("records with authors:", len(rec), "| unique:", rec["group"].nunique())

records with authors: 4484 | unique: 2734


In [5]:
# Scopus first: it carries the references for T0.4. Tracking goes last so that existing records are not altered
PRIORITY = {b: n for n, b in enumerate(DATABASES + list(T04_SOURCES))}
BLOCKS = {"S1": 0, "S2": 1, "tracking": 2}
IN_SCOPUS = ["scopus"] + list(T04_SOURCES)

def first_non_empty(series):
    return next((v for v in series if v), "")

rows = []
for _, g in rec.sort_values("database", key=lambda s: s.map(PRIORITY)).groupby("group", sort=False):
    rows.append({
        "title": g["title"].iloc[0], "authors": first_non_empty(g["authors"]),
        "year": first_non_empty(g["year"]), "source": first_non_empty(g["source"]),
        "type": first_non_empty(g["type"]), "language": first_non_empty(g["language"]),
        "doi": first_non_empty(g["doi"]),
        "abstract": max(g["abstract"], key=len),
        "keywords": "; ".join(v for v in g["keywords"] if v),
        "databases": "+".join(sorted(set(g["database"]), key=PRIORITY.get)),
        "blocks": "+".join(sorted(set(g["block"]), key=BLOCKS.get)),
        "id_scopus": first_non_empty(g.loc[g["database"].isin(IN_SCOPUS), "db_id"]),
        "id_wos": first_non_empty(g.loc[g["database"] == "wos", "db_id"]),
        "id_ieee": first_non_empty(g.loc[g["database"] == "ieee", "db_id"]),
        "title_key": g["title_key"].iloc[0],
    })
corpus = pd.DataFrame(rows)

# Groups of possible redundant versions
rep = corpus[corpus["title_key"].str.len() >= 25].groupby("title_key")["title"].transform("size") > 1
codes = {c: f"V{n:03d}" for n, c in enumerate(sorted(corpus.loc[rep[rep].index, "title_key"].unique()), 1)}
corpus["possible_version"] = corpus["title_key"].where(corpus.index.isin(rep[rep].index), "").map(lambda c: codes.get(c, ""))
print("records in possible-version groups:", (corpus["possible_version"] != "").sum(), "in", len(codes), "groups")

records in possible-version groups: 103 in 50 groups


## 4. Control set

Each control work is looked for in the exports of each database and of each block. K01–K24 check block S1 and K25–K27 check block S2.

In [6]:
control = pd.read_csv(CONTROL, dtype=str).fillna("")
control["doi"] = control["doi"].str.lower()

def retrieved(df, k):
    if k["doi"]:
        return (df["doi"] == k["doi"]).any()
    return df["title"].str.lower().str.contains(k["title_contains"], regex=False).any()

found = control[["id", "reference"]].copy()
for column, mask in [*[(d, raw["database"] == d) for d in DATABASES], ("S1", raw["block"] == "S1"),
                     ("S2", raw["block"] == "S2"), ("corpus", raw.index == raw.index)]:
    found[column] = [retrieved(raw[mask], k) for _, k in control.iterrows()]
expected = found["id"].map(lambda i: "S2" if int(i[1:]) >= 25 else "S1")
print("Retrieved by their block:", int(sum(found.at[i, b] for i, b in expected.items())), "of", len(found))
table(found[~found[DATABASES + ["S1", "S2"]].all(axis=1)].replace({True: "yes", False: "—"}))

Retrieved by their block: 25 of 27


id,reference,scopus,wos,ieee,S1,S2,corpus
K01,"Chakrabarti and Chatterjee, ICCAD 2000",yes,yes,yes,yes,—,yes
K02,"Huang and Cheng, ITC 1997",yes,yes,yes,yes,—,yes
K03,"Huang and Cheng, TCAS-II 2000",yes,yes,yes,yes,—,yes
K04,"Tadeusiewicz and Hałgas, JETTA 2017",—,—,—,—,—,yes
K05,"Brosa and Figueras, ETW 1999",yes,—,yes,yes,—,yes
K06,"Zilch, Lubaszewski and Balen, AICSP 2022",yes,yes,—,yes,—,yes
K07,"Tang et al., IEEE TCAD 2018",yes,yes,yes,yes,—,yes
K08,"Spinks et al., JETTA 2004",yes,yes,—,yes,—,yes
K09,"Grzechca, ECCTD 2011",yes,—,yes,yes,—,yes
K10,"Yang et al., EAAI 2026",yes,yes,—,yes,—,yes


## 5. Identifiers and output files

Identifiers are assigned by year and title. If `corpus.csv` already exists, every record keeps its own and new ones get the following numbers. A record is recognised by its DOI or, if it had none before, by title and year.

In [7]:
corpus["key_t"] = "t:" + corpus["title_key"] + ":" + corpus["year"]
corpus["key"] = corpus["doi"].where(corpus["doi"] != "", corpus["key_t"])
corpus = corpus.sort_values(["year", "title_key", "doi"]).reset_index(drop=True)

previous, earlier = {}, set()
if CORPUS.exists():
    old = pd.read_csv(CORPUS, dtype=str).fillna("")
    previous = dict(zip(old["key"], old["id"]))
    earlier = set(old["id"])
next_n = max([int(i[1:]) for i in previous.values()], default=0) + 1
ids = []
for k, kt in zip(corpus["key"], corpus["key_t"]):
    rid = previous.get(k) or previous.get(kt)
    if rid is None:
        rid = f"R{next_n:04d}"
        next_n += 1
    ids.append(rid)
corpus["id"] = ids
assert corpus["id"].is_unique
lost = earlier - set(ids)
assert not lost, f"identifiers that disappear: {sorted(lost)[:10]}"
print("New records with respect to the previous corpus:", len(set(ids) - earlier))

columns = ["id", "databases", "blocks", "year", "title", "authors", "source", "type", "language", "doi",
           "possible_version", "abstract", "keywords", "id_scopus", "id_wos", "id_ieee", "key"]
corpus.sort_values("id")[columns].to_csv(CORPUS, index=False, encoding="utf-8")

in_s1 = corpus["blocks"].str.contains("S1")
new_s2 = corpus["blocks"].str.contains("S2") & ~in_s1
NAMES = {"scopus": "Scopus", "wos": "Web of Science", "ieee": "IEEE Xplore"}
counts = pd.DataFrame([
    *[(f"{NAMES[d]} {q}", int(((raw["database"] == d) & (raw["query"] == q)).sum()))
      for d, q, _, _ in QUERIES if f"{d} {q}" not in pending],
    *[(f"Tracking: {database.replace('_', ' ')}", int((raw["database"] == database).sum())) for database in T04_SOURCES],
    ("Total identified", len(raw)),
    ("Removed: no authors (front matter, proceedings, standards)", int(front_matter.sum())),
    ("Removed: document type (tracking)", int(other_type.sum())),
    ("Removed: duplicates", len(rec) - len(corpus)),
    ("Corpus for screening", len(corpus)),
    ("  of which, from block S1", int(in_s1.sum())),
    ("  of which, from block S2 only", int(new_s2.sum())),
    ("  of which, from tracking only", int((corpus["blocks"] == "tracking").sum())),
], columns=["item", "records"])
counts.to_csv(COUNTS, index=False, encoding="utf-8")
if new_s2.sum() > S2_LIMIT:
    print(f"WARNING: S2 adds {new_s2.sum()} new records, above the protocol limit of {S2_LIMIT}")
table(counts)

New records with respect to the previous corpus: 0


item,records
Scopus S1,1380
Web of Science S1,801
IEEE Xplore X1,545
IEEE Xplore X2,488
Scopus S2,94
Web of Science S2,52
Web of Science S2_original,48
IEEE Xplore X3,12
IEEE Xplore X4,31
Tracking: backward citations,0


## 6. Corpus profile

In [8]:
print("By combination of blocks")
print(corpus["blocks"].value_counts().to_string(), "\n")
print("By combination of databases")
print(corpus["databases"].value_counts().to_string(), "\n")
decade = (pd.to_numeric(corpus["year"], errors="coerce") // 10 * 10).astype("Int64")
print("By decade")
print(decade.value_counts().sort_index().to_string(), "\n")
print("No abstract:", (corpus["abstract"].str.len() < 50).sum())
print("No DOI:", (corpus["doi"] == "").sum())
print("Language other than English (according to Scopus or WoS):",
      (~corpus["language"].str.lower().isin(["", "english", "undefined"])).sum())

By combination of blocks
blocks
S1                1549
tracking           747
S1+tracking        323
S2                 102
S1+S2                5
S1+S2+tracking       5
S2+tracking          3 

By combination of databases
databases
scopus                                      514
ieee                                        378
author                                      368
forward_citations                           365
scopus+wos                                  283
scopus+wos+ieee                             211
scopus+ieee                                 159
scopus+wos+forward_citations                102
wos                                          91
scopus+wos+ieee+forward_citations            87
scopus+forward_citations                     42
wos+forward_citations                        32
wos+ieee                                     20
scopus+ieee+forward_citations                19
forward_citations+author                     14
ieee+forward_citations                        9

## Conclusions and notes

- **Corpus for screening: 2734 records.** 1882 come from block S1, 105 only from block S2 and 747 only from the citation and author tracking of T0.4. Of 4565 records identified, 74 without authors, 7 tracking records by document type and 1750 duplicates are removed.
- **Block S2 is small and adds little overlap.** Its five queries give 237 records; only 10 of the S2 records were already retrieved by S1, so the two blocks cover different literatures. The 105 records new to S1 are well below the limit of 1500, so the protocol rule does not apply.
- **New identifiers: R2633 to R2734** (102 records). The other 3 records of S2 were already in the corpus through tracking. The new records need the first pass of T0.3.
- **Control set: 25 of 27 retrieved by their block.** S1 retrieves 22 of K01–K24 (K04 enters through tracking and K15 is still missing); S2 retrieves K25–K27.
- **Possible redundant versions** are flagged, not merged, and decided at full text (reason E6).

### Notes

- Web of Science has two S2 queries: the protocol string (`S2_original`, 48 records) and the variant WoS suggested (`S2`, 52 records), which has `fmeca` and `p242r` in place of `fmeda` and `P2427`. They share 44 records; the 4 found only by the protocol string are also in Scopus, and the 8 found only by the variant are FMECA reliability studies outside the field. The corpus is the same with or without the variant.
- Many S2 records are about battery management systems as a whole (state estimation, thermal runaway, communication); they are expected to be excluded by E3 at screening.